In [ ]:
from dotenv import load_dotenv
import os
import kagglehub
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    average_precision_score
)
from Preprocessor import preprocess_amex

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [16]:
def logistic_regression(data):
    # Features and target
    X = data.drop(columns=["Class"])
    y = data["Class"]

    # 80/20 train-test split
    # stratify=y preserves the fraud/non-fraud ratio in both sets
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.20,
        stratify=y,
        random_state=42
    )

    print(f"Training samples: {len(X_train)}")
    print(f"Test samples:     {len(X_test)}")
    print(f"Train fraud:      {y_train.sum()} ({y_train.mean():.4%})")
    print(f"Test fraud:       {y_test.sum()} ({y_test.mean():.4%})")


    # Scaling + Logistic Regression
    model = Pipeline([
        ("scaler", StandardScaler()),
        ("logreg", LogisticRegression(
            class_weight="balanced",
            max_iter=2000,
            random_state=42
        ))
    ])

    # Train
    model.fit(X_train, y_train)

    # Evaluate
    y_prob = model.predict_proba(X_test)[:, 1]
    y_pred = model.predict(X_test)

    auprc = average_precision_score(y_test, y_prob)
    auroc = roc_auc_score(y_test, y_prob)

    print(f"\nAUPRC: {auprc:.4f}")
    print(f"AUROC: {auroc:.4f}")
    print(f"Random AUPRC baseline: {y_test.mean():.4f}")

    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

In [17]:
#Use .env file with Kaggle API Token to download MLG ULB dataset
load_dotenv()
os.environ["KAGGLE_API_TOKEN"] = os.getenv("KAGGLE_API_TOKEN")
path = kagglehub.dataset_download("mlg-ulb/creditcardfraud")
file = os.path.join(path, 'creditcard.csv')
data = pd.read_csv(file)

print(f"Size of dataset: {len(data)}")
print(f"Number of features: {data.shape[1]}")
print(f"# of Frauds = {np.sum(data['Class'])}")
print(np.sum(data['Class'])/len(data['Class'])*100, '%')

Size of dataset: 284807
Number of features: 31
# of Frauds = 492
0.1727485630620034 %


In [18]:
print("ULB MLG credit card fraud data:\n")
logistic_regression(data)

ULB MLG credit card fraud data:

Training samples: 227845
Test samples:     56962
Train fraud:      394 (0.1729%)
Test fraud:       98 (0.1720%)

AUPRC: 0.7190
AUROC: 0.9721
Random AUPRC baseline: 0.0017

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.98      0.99     56864
           1       0.06      0.92      0.11        98

    accuracy                           0.98     56962
   macro avg       0.53      0.95      0.55     56962
weighted avg       1.00      0.98      0.99     56962



In [19]:
#Use .env file with Kaggle API Token to download AMEX dataset
load_dotenv()
os.environ["KAGGLE_API_TOKEN"] = os.getenv("KAGGLE_API_TOKEN")
path = kagglehub.competition_download("amex-default-prediction")
file = os.path.join(path, "train_data.csv")
data_all = pd.read_csv(file, nrows=1e6)
file = os.path.join(path, "train_labels.csv")
labels_all = pd.read_csv(file,nrows=1e6)

#Scrape all data associated with the a random n customers
customers = data_all["customer_ID"].drop_duplicates().sample(n=12000, random_state=42)
data_without_labels = data_all[data_all["customer_ID"].isin(customers)]
labels = labels_all[labels_all["customer_ID"].isin(customers)]
amex_df = preprocess_amex(data_without_labels, labels)
frac_fraud = np.sum(data['Class'])/len(data['Class'])

print(f"Size of dataset: {len(data)}")
print(f"Number of features: {data.shape[1]}")
print(f"# of Frauds = {int(np.sum(data['Class']))}")
print(frac_fraud*100, '% fraud')

Size of dataset: 284807
Number of features: 31
# of Frauds = 492
0.1727485630620034 % fraud


In [20]:
train_size = 2000
experiment_data, inference_data = train_test_split(amex_df, test_size=1 - train_size / len(amex_df), stratify=amex_df['Class'], random_state=42)
print("AMEX dataset LR:\n")
logistic_regression(amex_df)

AMEX dataset LR:

Training samples: 9600
Test samples:     2400
Train fraud:      2499.0 (26.0312%)
Test fraud:       625.0 (26.0417%)

AUPRC: 0.8656
AUROC: 0.9498
Random AUPRC baseline: 0.2604

Classification Report:
              precision    recall  f1-score   support

         0.0       0.96      0.87      0.91      1775
         1.0       0.70      0.90      0.79       625

    accuracy                           0.88      2400
   macro avg       0.83      0.89      0.85      2400
weighted avg       0.90      0.88      0.88      2400

